In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import cv2
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from tqdm import tqdm
import os
from PIL import Image
from torchvision.utils import save_image

Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Image Transformations - Consistent 256x256 resolution
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

Custom Dataset Loader

In [ ]:

class DeblurringDataset(Dataset):
    def __init__(self, blurry_dir, sharp_dir, transform=None):
        self.blurry_dir = blurry_dir
        self.sharp_dir = sharp_dir
        self.transform = transform
        self.image_filenames = sorted(os.listdir(blurry_dir))

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        blurry_path = os.path.join(self.blurry_dir, self.image_filenames[idx])
        sharp_path = os.path.join(self.sharp_dir, self.image_filenames[idx])

        blurry_img = Image.open(blurry_path).convert("RGB")
        sharp_img = Image.open(sharp_path).convert("RGB")

        if self.transform:
            blurry_img = self.transform(blurry_img)
            sharp_img = self.transform(sharp_img)

        return blurry_img, sharp_img

In [ ]:
# Create DataLoaders
blurry_train_dir = '/content/drive/My Drive/GOPRO_Large/train/blur'
sharp_train_dir = '/content/drive/My Drive/GOPRO_Large/train/sharp'
blurry_test_dir = '/content/drive/My Drive/GOPRO_Large/test/blur'
sharp_test_dir = '/content/drive/My Drive/GOPRO_Large/test/sharp'

train_dataset = DeblurringDataset(blurry_train_dir, sharp_train_dir, transform)
test_dataset = DeblurringDataset(blurry_test_dir, sharp_test_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

Generator Implementation

In [ ]:
# MobileViT Components
def conv_2d(inp, oup, kernel_size=3, stride=1, padding=0, groups=1, bias=False, norm=True, act=True):
    conv = nn.Sequential()
    conv.add_module('conv', nn.Conv2d(inp, oup, kernel_size, stride, padding, bias=bias, groups=groups))
    if norm:
        conv.add_module('BatchNorm2d', nn.BatchNorm2d(oup))
    if act:
        conv.add_module('Activation', nn.SiLU())
    return conv

In [ ]:
class InvertedResidual(nn.Module):
    def __init__(self, inp, oup, stride, expand_ratio):
        super(InvertedResidual, self).__init__()
        self.stride = stride
        assert stride in [1, 2]
        hidden_dim = int(round(inp * expand_ratio))
        self.block = nn.Sequential()
        if expand_ratio != 1:
            self.block.add_module('exp_1x1', conv_2d(inp, hidden_dim, kernel_size=1, stride=1, padding=0))
        self.block.add_module('conv_3x3', conv_2d(hidden_dim, hidden_dim, kernel_size=3, stride=stride, padding=1, groups=hidden_dim))
        self.block.add_module('red_1x1', conv_2d(hidden_dim, oup, kernel_size=1, stride=1, padding=0, act=False))
        self.use_res_connect = self.stride == 1 and inp == oup

    def forward(self, x):
        if self.use_res_connect:
            return x + self.block(x)
        else:
            return self.block(x)

In [ ]:
class LinearSelfAttention(nn.Module):
    def __init__(self, embed_dim, attn_dropout=0):
        super().__init__()
        self.qkv_proj = conv_2d(embed_dim, 1+2*embed_dim, kernel_size=1, bias=True, norm=False, act=False)
        self.attn_dropout = nn.Dropout(attn_dropout)
        self.out_proj = conv_2d(embed_dim, embed_dim, kernel_size=1, bias=True, norm=False, act=False)
        self.embed_dim = embed_dim

    def forward(self, x):
        qkv = self.qkv_proj(x)
        q, k, v = torch.split(qkv, split_size_or_sections=[1, self.embed_dim, self.embed_dim], dim=1)
        context_score = F.softmax(q, dim=-1)
        context_score = self.attn_dropout(context_score)
        context_vector = k * context_score
        context_vector = torch.sum(context_vector, dim=-1, keepdim=True)
        out = F.relu(v) * context_vector.expand_as(v)
        out = self.out_proj(out)
        return out

In [ ]:
class LinearAttnFFN(nn.Module):
    def __init__(self, embed_dim, ffn_latent_dim, dropout=0, attn_dropout=0):
        super().__init__()
        self.pre_norm_attn = nn.Sequential(
            nn.GroupNorm(num_channels=embed_dim, eps=1e-5, affine=True, num_groups=1),
            LinearSelfAttention(embed_dim, attn_dropout),
            nn.Dropout(dropout)
        )
        self.pre_norm_ffn = nn.Sequential(
            nn.GroupNorm(num_channels=embed_dim, eps=1e-5, affine=True, num_groups=1),
            conv_2d(embed_dim, ffn_latent_dim, kernel_size=1, stride=1, bias=True, norm=False, act=True),
            nn.Dropout(dropout),
            conv_2d(ffn_latent_dim, embed_dim, kernel_size=1, stride=1, bias=True, norm=False, act=False),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x + self.pre_norm_attn(x)
        x = x + self.pre_norm_ffn(x)
        return x

In [ ]:
class MobileViTBlockv3_v2(nn.Module):
    def __init__(self, inp, attn_dim, ffn_multiplier, attn_blocks, patch_size):
        super(MobileViTBlockv3_v2, self).__init__()
        self.patch_h, self.patch_w = patch_size
        self.local_rep = nn.Sequential()
        self.local_rep.add_module('conv_3x3', conv_2d(inp, inp, kernel_size=3, stride=1, padding=1, groups=inp))
        self.local_rep.add_module('conv_1x1', conv_2d(inp, attn_dim, kernel_size=1, stride=1, norm=False, act=False))

        self.global_rep = nn.Sequential()
        ffn_dims = [int((ffn_multiplier*attn_dim)//16*16)] * attn_blocks
        for i in range(attn_blocks):
            self.global_rep.add_module(f'LinearAttnFFN_{i}', LinearAttnFFN(attn_dim, ffn_dims[i]))
        self.global_rep.add_module('LayerNorm2D', nn.GroupNorm(num_channels=attn_dim, eps=1e-5, affine=True, num_groups=1))
        self.conv_proj = conv_2d(2*attn_dim, inp, kernel_size=1, stride=1, padding=0, act=False)

    def unfolding_pytorch(self, feature_map):
        batch_size, in_channels, img_h, img_w = feature_map.shape
        patches = F.unfold(feature_map, kernel_size=(self.patch_h, self.patch_w), stride=(self.patch_h, self.patch_w))
        patches = patches.reshape(batch_size, in_channels, self.patch_h * self.patch_w, -1)
        return patches, (img_h, img_w)

    def folding_pytorch(self, patches, output_size):
        batch_size, in_dim, patch_size, n_patches = patches.shape
        patches = patches.reshape(batch_size, in_dim * patch_size, n_patches)
        feature_map = F.fold(patches, output_size=output_size, kernel_size=(self.patch_h, self.patch_w), stride=(self.patch_h, self.patch_w))
        return feature_map

    def forward(self, x):
        res = x.clone()
        fm_conv = self.local_rep(x)
        x, output_size = self.unfolding_pytorch(fm_conv)
        x = self.global_rep(x)
        x = self.folding_pytorch(patches=x, output_size=output_size)
        x = self.conv_proj(torch.cat((x, fm_conv), dim=1))
        return x + res

In [ ]:
# Enhanced Generator with Spectral Normalization
class DeblurGeneratorMobileViT(nn.Module):
    def __init__(self, image_size=(256, 256), width_multiplier=1.0, patch_size=(2, 2)):
        super().__init__()
        ih, iw = image_size
        self.ph, self.pw = patch_size
        assert ih % self.ph == 0 and iw % self.pw == 0

        channels = [int(max(16, min(64, 32 * width_multiplier))),
                   int(64 * width_multiplier),
                   int(128 * width_multiplier),
                   int(256 * width_multiplier),
                   int(384 * width_multiplier),
                   int(512 * width_multiplier)]

        attn_dim = [int(128 * width_multiplier),
                   int(192 * width_multiplier),
                   int(256 * width_multiplier)]

        ffn_multiplier = 2
        mv2_exp_mult = 2

        # Encoder
        self.conv_in = conv_2d(3, channels[0], kernel_size=3, stride=1, padding=1)

        self.encoder_stage1 = nn.Sequential(
            InvertedResidual(channels[0], channels[1], stride=1, expand_ratio=mv2_exp_mult)
        )
        self.encoder_stage2 = nn.Sequential(
            InvertedResidual(channels[1], channels[2], stride=2, expand_ratio=mv2_exp_mult),
            InvertedResidual(channels[2], channels[2], stride=1, expand_ratio=mv2_exp_mult)
        )
        self.encoder_stage3 = nn.Sequential(
            InvertedResidual(channels[2], channels[3], stride=2, expand_ratio=mv2_exp_mult),
            MobileViTBlockv3_v2(channels[3], attn_dim[0], ffn_multiplier, 2, patch_size=patch_size)
        )
        self.encoder_stage4 = nn.Sequential(
            InvertedResidual(channels[3], channels[4], stride=2, expand_ratio=mv2_exp_mult),
            MobileViTBlockv3_v2(channels[4], attn_dim[1], ffn_multiplier, 4, patch_size=patch_size)
        )
        self.encoder_stage5 = nn.Sequential(
            InvertedResidual(channels[4], channels[5], stride=2, expand_ratio=mv2_exp_mult),
            MobileViTBlockv3_v2(channels[5], attn_dim[2], ffn_multiplier, 3, patch_size=patch_size)
        )

        # Decoder with skip connections
        self.decoder_stage5 = nn.Sequential(
            conv_2d(channels[5], channels[4], kernel_size=1, stride=1),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        )
        self.decoder_stage4 = nn.Sequential(
            conv_2d(channels[4] * 2, channels[3], kernel_size=1, stride=1),
            MobileViTBlockv3_v2(channels[3], attn_dim[0], ffn_multiplier, 2, patch_size=patch_size),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        )
        self.decoder_stage3 = nn.Sequential(
            conv_2d(channels[3] * 2, channels[2], kernel_size=1, stride=1),
            MobileViTBlockv3_v2(channels[2], attn_dim[0]//2, ffn_multiplier, 1, patch_size=patch_size),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        )
        self.decoder_stage2 = nn.Sequential(
            conv_2d(channels[2] * 2, channels[1], kernel_size=1, stride=1),
            InvertedResidual(channels[1], channels[1], stride=1, expand_ratio=mv2_exp_mult),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        )
        self.decoder_stage1 = nn.Sequential(
            conv_2d(channels[1] * 2, channels[0], kernel_size=1, stride=1),
            InvertedResidual(channels[0], channels[0], stride=1, expand_ratio=mv2_exp_mult)
        )

        # Output
        self.conv_out = nn.Sequential(
            conv_2d(channels[0], 32, kernel_size=3, stride=1, padding=1),
            conv_2d(32, 3, kernel_size=1, stride=1, act=False)
        )
        self.use_residual = True

    def forward(self, x):
        input_img = x
        x1 = self.conv_in(x)
        x1 = self.encoder_stage1(x1)
        x2 = self.encoder_stage2(x1)
        x3 = self.encoder_stage3(x2)
        x4 = self.encoder_stage4(x3)
        x5 = self.encoder_stage5(x4)

        d5 = self.decoder_stage5(x5)
        d5 = torch.cat([d5, x4], dim=1)

        d4 = self.decoder_stage4(d5)
        d4 = torch.cat([d4, x3], dim=1)

        d3 = self.decoder_stage3(d4)
        d3 = torch.cat([d3, x2], dim=1)

        d2 = self.decoder_stage2(d3)
        d2 = torch.cat([d2, x1], dim=1)

        d1 = self.decoder_stage1(d2)
        out = self.conv_out(d1)

        if self.use_residual:
            out = out + input_img

        return torch.clamp(out, 0, 1)

Discriminator Implementation

In [ ]:
# Enhanced Discriminator with Spectral Normalization
class DeblurDiscriminatorMobileViT(nn.Module):
    def __init__(self, image_size=(256, 256), width_multiplier=0.75, patch_size=(2, 2), dropout=0.3):
        super().__init__()
        ih, iw = image_size
        self.ph, self.pw = patch_size

        channels = [int(max(16, min(64, 32 * width_multiplier))),
                   int(64 * width_multiplier),
                   int(128 * width_multiplier),
                   int(256 * width_multiplier)]

        attn_dim = [int(128 * width_multiplier)]
        ffn_multiplier = 2
        mv2_exp_mult = 2

        # Input layer with spectral norm
        self.conv_in = nn.utils.spectral_norm(
            nn.Conv2d(3, channels[0], kernel_size=4, stride=2, padding=1)
        )

        # Feature extraction with spectral norm and dropout
        self.features = nn.Sequential(
            nn.utils.spectral_norm(
                nn.Conv2d(channels[0], channels[1], kernel_size=4, stride=2, padding=1)
            ),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(dropout),

            nn.utils.spectral_norm(
                nn.Conv2d(channels[1], channels[2], kernel_size=4, stride=2, padding=1)
            ),
            nn.LeakyReLU(0.2),
            nn.Dropout2d(dropout),

            MobileViTBlockv3_v2(channels[2], attn_dim[0], ffn_multiplier, 2, patch_size=patch_size),
            nn.Dropout2d(dropout),

            nn.utils.spectral_norm(
                nn.Conv2d(channels[2], channels[3], kernel_size=4, stride=2, padding=1)
            ),
            nn.LeakyReLU(0.2)
        )

        # PatchGAN output
        self.out = nn.utils.spectral_norm(
            nn.Conv2d(channels[3], 1, kernel_size=1, stride=1)
        )

    def forward(self, x):
        x = self.conv_in(x)
        x = self.features(x)
        x = self.out(x)
        return x

Loss Calculations

In [ ]:
# Enhanced Perceptual Loss using VGG16
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = torchvision.models.vgg16(pretrained=True).features[:16].eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg
        self.criterion = nn.L1Loss()

    def forward(self, fake, real):
        fake_vgg = self.vgg(fake)
        real_vgg = self.vgg(real)
        return self.criterion(fake_vgg, real_vgg)

In [ ]:
# Enhanced Wasserstein Loss with proper perceptual loss
class WassersteinLoss(nn.Module):
    def __init__(self, lambda_l1=100.0, lambda_perceptual=10.0, lambda_gp=10.0):
        super().__init__()
        self.l1_loss = nn.L1Loss()
        self.lambda_l1 = lambda_l1
        self.lambda_perceptual = lambda_perceptual
        self.lambda_gp = lambda_gp
        self.perceptual_loss = PerceptualLoss().to(device)

    def gradient_penalty(self, discriminator, real_samples, fake_samples):
        batch_size = real_samples.size(0)
        alpha = torch.rand(batch_size, 1, 1, 1, device=real_samples.device)
        interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
        d_interpolates = discriminator(interpolates)
        fake = torch.ones(d_interpolates.size(), device=real_samples.device, requires_grad=False)

        gradients = torch.autograd.grad(
            outputs=d_interpolates,
            inputs=interpolates,
            grad_outputs=fake,
            create_graph=True,
            retain_graph=True,
            only_inputs=True,
        )[0]

        gradients = gradients.view(batch_size, -1)
        gradient_norm = gradients.norm(2, dim=1)
        gradient_penalty = ((gradient_norm - 1) ** 2).mean()
        return gradient_penalty

    def generator_loss(self, fake_pred, fake_img, real_img):
        gan_loss = -torch.mean(fake_pred)
        l1_loss = self.l1_loss(fake_img, real_img) * self.lambda_l1
        percep_loss = self.perceptual_loss(fake_img, real_img) * self.lambda_perceptual

        return gan_loss + l1_loss + percep_loss, {
            'G_wasserstein': gan_loss.item(),
            'G_l1': l1_loss.item(),
            'G_percep': percep_loss.item()
        }

    def discriminator_loss(self, real_pred, fake_pred, discriminator, real_samples, fake_samples):
        w_loss = torch.mean(fake_pred) - torch.mean(real_pred)
        gp = self.gradient_penalty(discriminator, real_samples, fake_samples) * self.lambda_gp
        d_loss = w_loss + gp

        return d_loss, {
            'D_wasserstein': w_loss.item(),
            'D_gp': gp.item()
        }

In [ ]:
class EnhancedWassersteinLoss(WassersteinLoss):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.vgg = torchvision.models.vgg19(pretrained=True).features[:16].eval()
        for param in self.vgg.parameters():
            param.requires_grad = False

    def feature_matching_loss(self, fake, real):
        fake_features = self.vgg(fake)
        real_features = self.vgg(real)
        return F.l1_loss(fake_features, real_features)

    def generator_loss(self, fake_pred, fake_img, real_img):
        gan_loss = -torch.mean(fake_pred)
        l1_loss = self.l1_loss(fake_img, real_img) * self.lambda_l1
        percep_loss = self.perceptual_loss(fake_img, real_img) * self.lambda_perceptual
        fm_loss = self.feature_matching_loss(fake_img, real_img) * 5.0  # New weight

        total_loss = gan_loss + l1_loss + percep_loss + fm_loss
        metrics = {
            'G_wasserstein': gan_loss.item(),
            'G_l1': l1_loss.item(),
            'G_percep': percep_loss.item(),
            'G_fm': fm_loss.item()
        }
        return total_loss, metrics

Initializations

In [ ]:
# Initialize models
generator = DeblurGeneratorMobileViT(image_size=(256, 256), width_multiplier=1.0).to(device)
discriminator = DeblurDiscriminatorMobileViT(image_size=(256, 256), width_multiplier=0.75).to(device)

# Weight initialization
def weights_init(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)

generator.apply(weights_init)
discriminator.apply(weights_init)

# Optimizers with different learning rates
opt_G = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.9))
opt_D = optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.9))

# Learning rate schedulers
g_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_G, mode='min', factor=0.5, patience=5, verbose=True)
d_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_D, mode='min', factor=0.5, patience=5, verbose=True)

# Loss function
criterion = WassersteinLoss(lambda_l1=10.0, lambda_perceptual=10.0, lambda_gp=10.0).to(device)

Checkpoint handling

In [ ]:

def save_checkpoint(epoch):
    torch.save({
        'epoch': epoch,
        'generator_state_dict': generator.state_dict(),
        'discriminator_state_dict': discriminator.state_dict(),
        'opt_G_state_dict': opt_G.state_dict(),
        'opt_D_state_dict': opt_D.state_dict(),
    }, f'/content/drive/MyDrive/checkpoints/deblur_checkpoints_epoch_{epoch}.pth')

def load_checkpoint(path):
    checkpoint = torch.load(path, map_location=device)
    generator.load_state_dict(checkpoint['generator_state_dict'])
    discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
    opt_G.load_state_dict(checkpoint['opt_G_state_dict'])
    opt_D.load_state_dict(checkpoint['opt_D_state_dict'])
    return checkpoint['epoch']

Training Step

In [ ]:
import time
import datetime
import os

checkpoint_path = '/content/drive/MyDrive/checkpoints/deblur_checkpoints_epoch_1.pth'
start_epoch = 0

# Load checkpoint if available
if os.path.exists(checkpoint_path):
    start_epoch = load_checkpoint(checkpoint_path)
    print(f"Resuming training from epoch {start_epoch}")

# Training loop
def train(generator, discriminator, train_loader, opt_G, opt_D, criterion, epochs=200):
    generator.train()
    discriminator.train()

    bat_per_epo = len(train_loader)  # Batches per epoch
    n_steps = bat_per_epo * epochs   # Total training steps

    print(f"No. of batches per epoch: {bat_per_epo}")
    print(f"[!!] Total training steps: {n_steps}")

    total_steps = start_epoch * bat_per_epo

    for epoch in range(start_epoch, epochs + 1):
        epoch_start_time = time.time()  # Start time for the entire epoch
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')

        for i, (blurry, sharp) in enumerate(progress_bar):
            blurry, sharp = blurry.to(device), sharp.to(device)
            batch_start_time = time.time()

            # --- Discriminator Update (every 5 steps) ---
            if i % 5 == 0:
                opt_D.zero_grad()
                with torch.no_grad():
                    fake_sharp = generator(blurry)

                real_pred = discriminator(sharp)
                fake_pred = discriminator(fake_sharp.detach())

                d_loss, d_metrics = criterion.discriminator_loss(real_pred, fake_pred, discriminator, sharp, fake_sharp)
                d_loss.backward()
                opt_D.step()

            # --- Generator Update ---
            opt_G.zero_grad()
            fake_sharp = generator(blurry)
            fake_pred = discriminator(fake_sharp)

            g_loss, g_metrics = criterion.generator_loss(fake_pred, fake_sharp, sharp)
            g_loss.backward()
            opt_G.step()

            # Update progress bar
            progress_bar.set_postfix({
                'D_loss': d_metrics['D_wasserstein'],
                'G_loss': g_metrics['G_wasserstein'],
                'GP': d_metrics['D_gp'],
                'L1': g_metrics['G_l1']
            })

        # --- Epoch Logging ---
        epoch_time = time.time() - epoch_start_time
        total_steps += bat_per_epo  # Increment by batches processed this epoch
        remaining_steps = n_steps - total_steps

        # Calculate remaining time based on average time per batch
        avg_time_per_batch = epoch_time / bat_per_epo
        estimated_time_left = avg_time_per_batch * remaining_steps

        print(f"--- [Epoch Time: {epoch_time:.2f} s] [Time Left: {str(datetime.timedelta(seconds=int(estimated_time_left)))}]-----")

        # Update schedulers
        g_scheduler.step(g_loss)
        d_scheduler.step(d_loss)

        # --- Checkpointing ---
        if (epoch + 1) % 10 == 0:
            save_checkpoint(epoch + 1)
            with torch.no_grad():
                sample = generator(blurry[:1])
                save_image(
                    torch.cat([blurry[:1], sample, sharp[:1]], dim=0),
                    f'/content/drive/MyDrive/samples/epoch_{epoch+1}.png',
                    normalize=True)

# Start training
train(generator, discriminator, train_loader, opt_G, opt_D, criterion, epochs=200)

In [ ]:
def load_checkpoint(path):
    checkpoint = torch.load(path, map_location=device)
    generator = (checkpoint['generator_state_dict'])
    return generator

In [ ]:
generator = load_checkpoint('/content/drive/MyDrive/checkpoints/deblur_checkpoints_epoch_200.pth')

Model Saving

In [ ]:
# Save generator in inference-ready format
torch.save({
    'model_state_dict': generator,
    'image_size': (256, 256),
    'width_multiplier': 1.0,
    'patch_size': (2, 2),
}, '/content/drive/MyDrive/checkpoints/generator.pth')

Verification

In [ ]:
# Verification function
def verify_saved_model():
    # Load checkpoint
    checkpoint = torch.load('/content/drive/MyDrive/checkpoints/generator.pth', map_location=device)

    # Recreate model architecture
    loaded_generator = DeblurGeneratorMobileViT(
        image_size=checkpoint['image_size'],
        width_multiplier=checkpoint['width_multiplier'],
        patch_size=checkpoint['patch_size']
    ).to(device)

    # Load weights
    loaded_generator.load_state_dict(checkpoint['model_state_dict'])
    loaded_generator.eval()

    # Test inference
    with torch.no_grad():
        test_input = torch.rand(1, 3, 256, 256).to(device)
        output = loaded_generator(test_input)
        print("Verification successful! Output shape:", output.shape)
        return loaded_generator

verified_generator = verify_saved_model()

In [ ]:
def calculate_ssim(img1, img2):
    """
    Calculate SSIM with automatic window size adjustment
    """
    # Convert tensors to numpy arrays if needed
    if isinstance(img1, torch.Tensor):
        img1 = img1.cpu().numpy().transpose(1, 2, 0)
    if isinstance(img2, torch.Tensor):
        img2 = img2.cpu().numpy().transpose(1, 2, 0)

    # Ensure values are in [0, 1]
    img1 = np.clip(img1, 0, 1)
    img2 = np.clip(img2, 0, 1)

    # Determine appropriate window size
    min_dim = min(img1.shape[:2])
    win_size = min(7, min_dim)  # Default is 7, but adjust if image is smaller
    win_size = win_size if win_size % 2 == 1 else win_size - 1  # Ensure odd

    # Calculate SSIM
    return ssim(img1, img2, win_size=win_size, channel_axis=-1, data_range=1.0)

def evaluate(verified_generator, test_loader):
    verified_generator.eval()
    psnr_values = []
    ssim_values = []

    with torch.no_grad():
        for blurry, sharp in tqdm(test_loader, desc='Evaluating'):
            blurry = blurry.to(device)
            fake_sharp = verified_generator(blurry).cpu()

            for i in range(fake_sharp.shape[0]):
                # Get single image pair
                fake_img = fake_sharp[i]
                real_img = sharp[i]

                # Calculate metrics
                try:
                    psnr_val = psnr(real_img.numpy(), fake_img.numpy(), data_range=1.0)
                    ssim_val = calculate_ssim(real_img, fake_img)  # Use our safe function

                    psnr_values.append(psnr_val)
                    ssim_values.append(ssim_val)
                except ValueError as e:
                    print(f"Skipping image {i} due to error: {str(e)}")
                    continue

    print(f'\nAverage PSNR: {np.mean(psnr_values):.2f} dB')
    print(f'Average SSIM: {np.mean(ssim_values):.4f}')
    return psnr_values, ssim_values

# Run evaluation
psnr_vals, ssim_vals = evaluate(verified_generator, test_loader)

In [ ]:
from matplotlib import pyplot as plt
# Visualization
def visualize_results(verified_generator, test_dataset, num_samples=3):
    indices = np.random.choice(len(test_dataset), num_samples, replace=False)
    plt.figure(figsize=(15, 10))

    for i, idx in enumerate(indices):
        blurry, sharp = test_dataset[idx]
        blurry = blurry.unsqueeze(0).to(device)

        with torch.no_grad():
            fake_sharp = verified_generator(blurry).cpu().squeeze()

        blurry = blurry.cpu().squeeze()
        sharp = sharp.squeeze()

        # Calculate metrics for this sample
        fake_np = np.clip(fake_sharp.permute(1, 2, 0).numpy(), 0, 1)
        sharp_np = np.clip(sharp.permute(1, 2, 0).numpy(), 0, 1)
        psnr_val = psnr(sharp_np, fake_np, data_range=1.0)
        ssim_val = ssim(sharp_np, fake_np, data_range=1.0, multichannel=True)

        # Plot
        plt.subplot(3, num_samples, i+1)
        plt.imshow(blurry.permute(1, 2, 0))
        plt.title('Blurry Input')
        plt.axis('off')

        plt.subplot(3, num_samples, i+1+num_samples)
        plt.imshow(sharp.permute(1, 2, 0))
        plt.title('Ground Truth')
        plt.axis('off')

        plt.subplot(3, num_samples, i+1+2*num_samples)
        plt.imshow(fake_sharp.permute(1, 2, 0))
        plt.title(f'Deblurred\nPSNR: {psnr_val:.2f} dB')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Visualize results
visualize_results(verified_generator, test_dataset)